# PND 2025 | Camada GOLD | `gold.pnd_2025_proeficiencia`

**Objetivo:** gerar a tabela analítica de proficiência média por município, grupo e caderno, a partir da silver.

| Item | Definição |
|---|---|
| **Origem** | `silver.microdados2025_pnd_arq01` |
| **Destino** | `gold.pnd_2025_proeficiencia` |
| **Granularidade** | Uma linha por `nu_ano` + `co_grupo` + `co_municipio_prova` + `co_caderno` |
| **Chave primária** | `nu_ano`, `co_grupo`, `co_municipio_prova`, `co_caderno` |
| **Medidas** | `media_proficiencia` = média de `proficiencia`; `media_qt_acertos` = média de `qt_acertos` |

**Por que não há cruzamento entre arquivos:** os arquivos `arq2` a `arq13` só têm `nu_ano`, `co_grupo` e as respostas do caderno,
sem campo que permita ligar as linhas ao `arq01`. Por isso a gold usa **apenas** o `arq01` e as agregações ficam no nível de grupo.

**Regras aplicadas**
1. Mantém apenas as colunas necessárias (chaves e medidas). As demais são descartadas.
2. Converte as medidas para numérico de forma segura (valor inválido vira nulo, sem derrubar o job).
3. Exclui linhas com **chave nula** (uma chave primária não aceita nulos). A quantidade excluída é registrada.
4. Agrega por chave e calcula as médias. Valores nulos **não entram** na média (comportamento padrão do `AVG`).
5. Recria a tabela, define as chaves como `NOT NULL` e declara a chave primária.

**Fluxo**
```
Preparação (1-5) → Leitura e diagnóstico (6-9) → Tratamento (10-12)
   → Agregação (13-14) → Gravação (15-19) → Validação (20-22)
```

> Execute as células **uma a uma, em ordem**. O notebook é reexecutável: a tabela é recriada a cada execução.

## Passo 1 | Definir o catálogo

Mesmo parâmetro dos notebooks anteriores.

In [0]:
dbutils.widgets.text("catalog", "pnd2025", "Catálogo")

## Passo 2 | Ler o parâmetro

In [0]:
CATALOG = dbutils.widgets.get("catalog").lower()
print(f"Catálogo: {CATALOG}")

Catálogo: pnd2025


## Passo 3 | Selecionar o catálogo

As células `%sql` deste notebook usam nomes sem catálogo (ex.: `gold.pnd_2025_proeficiencia`) e dependem deste passo.
Se o Python for reiniciado, rode-o novamente.

In [0]:
spark.sql(f"USE CATALOG `{CATALOG}`")
display(spark.sql("SELECT current_catalog() AS catalogo_atual"))

catalogo_atual
pnd2025


## Passo 4 | Garantir que o schema gold existe

`IF NOT EXISTS` não faz nada se ele já existir.

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS gold
COMMENT 'Camada gold - tabelas analiticas e agregacoes para consumo'

## Passo 5 | Definir a especificação da tabela

Concentra em um só lugar tudo o que define a gold: tabelas de origem e destino, colunas-chave e colunas de medida.
Se algum nome de coluna da silver for diferente, **ajuste apenas esta célula**.

`AUDIT` é um dicionário que vai coletando métricas ao longo do notebook (linhas lidas, excluídas, geradas) e é exibido no passo 22.

In [0]:
# Origem e destino
SRC_TABLE = f"`{CATALOG}`.silver.microdados2025_pnd_arq01"
TGT_NAME = "pnd_2025_proeficiencia"
TGT_TABLE = f"`{CATALOG}`.gold.{TGT_NAME}"

# Chave primária (granularidade da tabela)
KEYS = ["nu_ano", "co_grupo", "co_municipio_prova", "co_caderno"]

# Colunas de medida na silver
COL_PROFICIENCIA = "proficiencia"
COL_ACERTOS = "qt_acertos"
MEASURES_SRC = [COL_PROFICIENCIA, COL_ACERTOS]

# Métricas de auditoria da execução
AUDIT = {}

print("Origem :", SRC_TABLE)
print("Destino:", TGT_TABLE)
print("Chave  :", KEYS)

Origem : `pnd2025`.silver.microdados2025_pnd_arq01
Destino: `pnd2025`.gold.pnd_2025_proeficiencia
Chave  : ['nu_ano', 'co_grupo', 'co_municipio_prova', 'co_caderno']


## Passo 6 | Ler a tabela da silver

Cria o DataFrame de origem e registra quantas linhas ele tem. Esta contagem é o ponto de partida para
reconciliar o que entra e o que sai da gold.

In [0]:
from pyspark.sql import functions as F

df_src = spark.table(SRC_TABLE)

AUDIT["linhas_silver"] = df_src.count()
print(f"Linhas na silver: {AUDIT['linhas_silver']:,}")

Linhas na silver: 760,106


## Passo 7 | Inspecionar o schema e uma amostra

Veja os **tipos** das colunas (o passo 10 depende disso) e confira se as colunas-chave e de medida estão onde esperado.

In [0]:
df_src.printSchema()
display(df_src.select(*KEYS, *MEASURES_SRC).limit(10))

root
 |-- nu_ano: integer (nullable = true)
 |-- co_grupo: integer (nullable = true)
 |-- co_municipio_prova: integer (nullable = true)
 |-- sg_uf_municipio_prova: string (nullable = true)
 |-- tp_inscricao_pnd: integer (nullable = true)
 |-- in_reaplicacao: integer (nullable = true)
 |-- co_caderno: integer (nullable = true)
 |-- tp_pres: integer (nullable = true)
 |-- tp_sit_disc: integer (nullable = true)
 |-- proficiencia: string (nullable = true)
 |-- nt_obj: string (nullable = true)
 |-- nt_dis: string (nullable = true)
 |-- nt_ger: string (nullable = true)
 |-- qt_acertos: string (nullable = true)
 |-- co_rs_i1: string (nullable = true)
 |-- co_rs_i2: string (nullable = true)
 |-- co_rs_i3: string (nullable = true)
 |-- co_rs_i4: string (nullable = true)
 |-- co_rs_i5: string (nullable = true)
 |-- co_rs_i6: string (nullable = true)
 |-- co_rs_i7: string (nullable = true)
 |-- co_rs_i8: string (nullable = true)
 |-- co_rs_i9: string (nullable = true)



nu_ano,co_grupo,co_municipio_prova,co_caderno,proficiencia,qt_acertos
2025,702,1100205,3,"-2,46117",0
2025,702,1100205,3,"-2,46117",0
2025,702,1200203,3,"-2,46117",0
2025,702,1501402,2,"-2,46117",0
2025,702,1702208,4,"-2,46117",0
2025,702,1702406,2,"-2,46117",0
2025,702,1705508,3,"-2,46117",0
2025,702,2208403,1,"-2,46117",0
2025,702,2211001,3,"-2,23714",2
2025,702,2211001,4,"-2,46117",0


## Passo 8 | Validar que as colunas necessárias existem

Falha logo no início, com mensagem clara, se alguma coluna esperada não estiver na silver.
Isso evita um erro obscuro mais adiante.

In [0]:
colunas_necessarias = KEYS + MEASURES_SRC
faltando = [c for c in colunas_necessarias if c not in df_src.columns]

assert not faltando, (
    f"Colunas ausentes na silver: {faltando}. "
    f"Colunas disponíveis: {df_src.columns}. Ajuste o passo 5."
)
print("Todas as colunas necessárias estão presentes.")

Todas as colunas necessárias estão presentes.


## Passo 9 | Perfilar nulos nas chaves e nas medidas

Mostra, para cada coluna usada, **quantas linhas estão nulas**. Leitura:

- **Nulos em chave:** essas linhas serão excluídas no passo 12 (chave primária não aceita nulo).
- **Nulos em medida:** não impedem a linha de existir, mas **não entram na média**.

In [0]:
perfil = df_src.select(
    F.count("*").alias("total_linhas"),
    *[F.sum(F.col(c).isNull().cast("int")).alias(f"nulos_{c}") for c in colunas_necessarias],
)
display(perfil)

total_linhas,nulos_nu_ano,nulos_co_grupo,nulos_co_municipio_prova,nulos_co_caderno,nulos_proficiencia,nulos_qt_acertos
760106,0,0,0,0,0,0


## Passo 10 | Selecionar colunas e converter as medidas para numérico

Mantém **somente** chaves e medidas; todas as outras colunas são descartadas aqui.

As medidas são convertidas para `double` com `try_cast`: se algum valor não for numérico, ele vira **nulo** em vez de
interromper o job (o Databricks serverless usa modo ANSI, em que um `cast` comum falharia). Se a coluna for texto,
a vírgula decimal é trocada por ponto antes da conversão. Se já for numérica, a conversão não altera nada.

In [0]:
def medida_como_double(nome: str):
    # Texto: troca vírgula decimal por ponto. Numérico: usa a coluna como está.
    tipo = dict(df_src.dtypes)[nome]
    origem = f"replace(trim(`{nome}`), ',', '.')" if tipo == "string" else f"`{nome}`"
    return F.expr(f"try_cast({origem} AS DOUBLE)").alias(nome)


df_typed = df_src.select(
    *[F.col(c) for c in KEYS],
    *[medida_como_double(m) for m in MEASURES_SRC],
)
df_typed.printSchema()

root
 |-- nu_ano: integer (nullable = true)
 |-- co_grupo: integer (nullable = true)
 |-- co_municipio_prova: integer (nullable = true)
 |-- co_caderno: integer (nullable = true)
 |-- proficiencia: double (nullable = true)
 |-- qt_acertos: double (nullable = true)



## Passo 11 | Auditar o efeito da conversão

Compara os nulos **antes** e **depois** da conversão. Se a coluna passar a ter mais nulos, existiam valores não numéricos
na silver, agora convertidos para nulo. Diferença `0` significa conversão limpa.

In [0]:
nulos_antes = df_src.select(*[F.sum(F.col(m).isNull().cast("int")).alias(m) for m in MEASURES_SRC]).first().asDict()
nulos_depois = df_typed.select(*[F.sum(F.col(m).isNull().cast("int")).alias(m) for m in MEASURES_SRC]).first().asDict()

for m in MEASURES_SRC:
    novos = nulos_depois[m] - nulos_antes[m]
    AUDIT[f"{m}_convertidos_para_nulo"] = novos
    print(f"{m}: nulos antes={nulos_antes[m]:,} | depois={nulos_depois[m]:,} | convertidos para nulo={novos:,}")

proficiencia: nulos antes=0 | depois=966 | convertidos para nulo=966
qt_acertos: nulos antes=0 | depois=966 | convertidos para nulo=966


## Passo 12 | Excluir linhas com chave nula

Uma chave primária não pode ter nulos, e uma linha sem município, grupo, ano ou caderno também não pode ser agrupada
de forma confiável. Estas linhas são removidas, e a **quantidade excluída fica registrada** em `AUDIT`.

In [0]:
from functools import reduce

# Condição: todas as colunas-chave preenchidas
chave_preenchida = reduce(lambda a, b: a & b, [F.col(c).isNotNull() for c in KEYS])

df_valid = df_typed.where(chave_preenchida)

AUDIT["linhas_validas"] = df_valid.count()
AUDIT["linhas_excluidas_chave_nula"] = AUDIT["linhas_silver"] - AUDIT["linhas_validas"]

print(f"Linhas válidas : {AUDIT['linhas_validas']:,}")
print(f"Linhas excluídas por chave nula: {AUDIT['linhas_excluidas_chave_nula']:,}")

Linhas válidas : 760,106
Linhas excluídas por chave nula: 0


## Passo 13 | Agregar por chave e calcular as médias

Agrupa por `nu_ano`, `co_grupo`, `co_municipio_prova` e `co_caderno` e calcula:

- `media_proficiencia` = média de `proficiencia`
- `media_qt_acertos` = média de `qt_acertos`

O `AVG` ignora nulos: a média considera apenas as linhas em que a medida está preenchida.

In [0]:
df_gold = (
    df_valid
    .groupBy(*KEYS)
    .agg(
        F.avg(COL_PROFICIENCIA).alias("media_proficiencia"),
        F.avg(COL_ACERTOS).alias("media_qt_acertos"),
    )
)

AUDIT["grupos_gold"] = df_gold.count()
print(f"Grupos (linhas da gold): {AUDIT['grupos_gold']:,}")
display(df_gold.orderBy(*KEYS).limit(10))

Grupos (linhas da gold): 39,203


nu_ano,co_grupo,co_municipio_prova,co_caderno,media_proficiencia,media_qt_acertos
2025,702,1100023,1,-0.47215,26.583333333333332
2025,702,1100023,2,-0.48550133333333334,26.266666666666666
2025,702,1100023,3,-0.16414285714285715,32.0
2025,702,1100023,4,-0.415915,27.428571428571427
2025,702,1100049,1,-0.09132599999999996,33.4
2025,702,1100049,2,-0.11032099999999993,33.1
2025,702,1100049,3,-0.1815983333333333,31.666666666666668
2025,702,1100049,4,-0.10255583333333333,33.166666666666664
2025,702,1100056,1,-0.21857,31.25
2025,702,1100056,2,0.14017333333333334,37.666666666666664


## Passo 14 | Validar a unicidade da chave antes de gravar

Confirma que **não há chave repetida** (nº de linhas = nº de chaves distintas) e que nenhuma chave está nula.
Como o agrupamento já garante isso, a verificação serve de rede de segurança e de evidência de auditoria.

In [0]:
chaves_distintas = df_gold.select(*KEYS).distinct().count()

assert chaves_distintas == AUDIT["grupos_gold"], (
    f"Chave duplicada: {AUDIT['grupos_gold']:,} linhas para {chaves_distintas:,} chaves distintas."
)
print(f"Unicidade OK: {chaves_distintas:,} chaves distintas.")

Unicidade OK: 39,203 chaves distintas.


## Passo 15 | Gravar a tabela gold

Remove a tabela anterior (se existir) e grava de novo. O `DROP` antes do `saveAsTable` evita que restrições
(`NOT NULL`, chave primária) de uma execução anterior atrapalhem a recriação. Como a gold é **derivada** da silver,
recriá-la é seguro.

In [0]:
spark.sql(f"DROP TABLE IF EXISTS {TGT_TABLE}")

(
    df_gold.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(TGT_TABLE)
)
print(f"Tabela gravada: {TGT_TABLE}")

Tabela gravada: `pnd2025`.gold.pnd_2025_proeficiencia


## Passo 16 | Documentar a tabela e as colunas

Comentários ficam visíveis no Catalog Explorer e ajudam quem for consumir a tabela.

In [0]:
spark.sql(
    f"COMMENT ON TABLE {TGT_TABLE} IS "
    "'PND 2025 - proficiencia media por ano, grupo, municipio de prova e caderno. "
    "Origem: silver.microdados2025_pnd_arq01'"
)

comentarios_colunas = {
    "nu_ano": "Ano de aplicacao",
    "co_grupo": "Codigo da area da prova de enquadramento do curso no Enade",
    "co_municipio_prova": "Codigo do municipio de realizacao da prova",
    "co_caderno": "Codigo do caderno de prova",
    "media_proficiencia": "Media da proficiencia (valores nulos nao entram na media)",
    "media_qt_acertos": "Media da quantidade de acertos (valores nulos nao entram na media)",
}

for coluna, texto in comentarios_colunas.items():
    spark.sql(f"COMMENT ON COLUMN {TGT_TABLE}.`{coluna}` IS '{texto}'")

print("Comentários aplicados.")

Comentários aplicados.


## Passo 17 | Definir as colunas-chave como `NOT NULL`

Pré-requisito para declarar a chave primária. Como o passo 12 já removeu as chaves nulas, o comando deve passar sem erro.

In [0]:
for c in KEYS:
    spark.sql(f"ALTER TABLE {TGT_TABLE} ALTER COLUMN `{c}` SET NOT NULL")
    print(f"NOT NULL aplicado: {c}")

NOT NULL aplicado: nu_ano
NOT NULL aplicado: co_grupo
NOT NULL aplicado: co_municipio_prova
NOT NULL aplicado: co_caderno


## Passo 18 | Declarar a chave primária

A chave primária no Unity Catalog é **informativa**: documenta o modelo e é usada por ferramentas de BI e de modelagem,
mas **não é imposta** pelo banco. Por isso a unicidade foi verificada nos passos 14 e 20.

Se este comando falhar na sua conta, a tabela continua válida; a falha só impede o registro da restrição.

In [0]:
colunas_pk = ", ".join(f"`{c}`" for c in KEYS)

try:
    spark.sql(f"ALTER TABLE {TGT_TABLE} ADD CONSTRAINT pk_{TGT_NAME} PRIMARY KEY ({colunas_pk})")
    print(f"Chave primária criada: pk_{TGT_NAME} ({colunas_pk})")
except Exception as e:
    print("Não foi possível criar a chave primária (a tabela segue válida).")
    print(f"Detalhe: {str(e)[:300]}")

Chave primária criada: pk_pnd_2025_proeficiencia (`nu_ano`, `co_grupo`, `co_municipio_prova`, `co_caderno`)


## Passo 19 | Conferir as restrições registradas

Consulta o `information_schema`. Deve aparecer uma restrição `PRIMARY KEY` para a tabela.

In [0]:
%sql
SELECT constraint_name, constraint_type
FROM information_schema.table_constraints
WHERE table_schema = 'gold'
  AND table_name = 'pnd_2025_proeficiencia'

constraint_name,constraint_type
pk_pnd_2025_proeficiencia,PRIMARY KEY


## Passo 20 | Validar a tabela gravada

Relê a tabela do disco e confirma:

1. O número de linhas gravadas é igual ao número de grupos calculados.
2. A chave continua única.
3. Nenhuma chave está nula.

In [0]:
df_check = spark.table(TGT_TABLE)

linhas_gravadas = df_check.count()
chaves_unicas = df_check.select(*KEYS).distinct().count()
chaves_nulas = df_check.where(~chave_preenchida).count()

AUDIT["linhas_gravadas_gold"] = linhas_gravadas

print(f"Linhas gravadas : {linhas_gravadas:,} (esperado {AUDIT['grupos_gold']:,})")
print(f"Chaves únicas   : {chaves_unicas:,}")
print(f"Chaves nulas    : {chaves_nulas:,}")

assert linhas_gravadas == AUDIT["grupos_gold"], "Divergência entre linhas calculadas e gravadas."
assert chaves_unicas == linhas_gravadas, "Há chaves duplicadas na tabela gravada."
assert chaves_nulas == 0, "Há chaves nulas na tabela gravada."
print("\nValidação concluída: tudo OK.")

Linhas gravadas : 39,203 (esperado 39,203)
Chaves únicas   : 39,203
Chaves nulas    : 0

Validação concluída: tudo OK.


## Passo 21 | Visualizar o resultado

Amostra da tabela final, ordenada pela chave.

In [0]:
%sql
SELECT *
FROM gold.pnd_2025_proeficiencia
ORDER BY nu_ano, co_grupo, co_municipio_prova, co_caderno
LIMIT 20

nu_ano,co_grupo,co_municipio_prova,co_caderno,media_proficiencia,media_qt_acertos
2025,702,1100023,1,-0.47215,26.583333333333332
2025,702,1100023,2,-0.48550133333333334,26.266666666666666
2025,702,1100023,3,-0.16414285714285715,32.0
2025,702,1100023,4,-0.415915,27.428571428571427
2025,702,1100049,1,-0.09132599999999996,33.4
2025,702,1100049,2,-0.11032099999999993,33.1
2025,702,1100049,3,-0.1815983333333333,31.666666666666668
2025,702,1100049,4,-0.10255583333333333,33.166666666666664
2025,702,1100056,1,-0.21857,31.25
2025,702,1100056,2,0.14017333333333334,37.666666666666664


## Passo 22 | Resumo de auditoria da execução

Reúne as métricas coletadas: de quantas linhas partimos, quantas foram descartadas e quantas linhas a gold tem.
Para o histórico de versões da tabela, use `DESCRIBE HISTORY gold.pnd_2025_proeficiencia`.

In [0]:
import pandas as pd

resumo = pd.DataFrame(
    [{"metrica": k, "valor": v} for k, v in AUDIT.items()]
)
display(resumo)

metrica,valor
linhas_silver,760106
proficiencia_convertidos_para_nulo,966
qt_acertos_convertidos_para_nulo,966
linhas_validas,760106
linhas_excluidas_chave_nula,0
grupos_gold,39203
linhas_gravadas_gold,39203


## Próximos passos

- Para **reagregar** estas médias em níveis maiores (ex.: por grupo, sem município), seria necessário guardar também a
  **quantidade de registros** de cada linha, pois a média de médias não é a média geral. Se esse caso de uso existir,
  vale incluir uma coluna de contagem.
- Ligar a gold a uma visão de consumo (dashboard ou consulta SQL) usando `co_municipio_prova` e `co_grupo` com as tabelas de apoio.